# Chai-1 Structure Prediction

This notebook demonstrates multi-modal structure prediction using Chai-1 from Chai Discovery. Chai-1 predicts the three-dimensional structures of proteins, ligands, and glycans, as well as their complexes, by combining ESM language model embeddings with a trunk network and diffusion-based structure generation. We demonstrate `run_chai1` by predicting the structure of the MfnG protein bound to L-tyrosine, covering input construction, model configuration, result inspection, visualization, and export.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("chai1")
display_overview("chai1")
display_docs_section("chai1", "Background")

# Chai1

Chai1 is a multi-modal structure prediction model from Chai Discovery that predicts 3D structures of proteins, ligands, and glycans using a [diffusion](https://en.wikipedia.org/wiki/Diffusion_model)-based architecture. It excels at modeling protein-ligand complexes and can incorporate ESM embeddings for improved accuracy.

## Background

**What does this tool measure/predict?**
Chai1 predicts the 3D atomic coordinates of biomolecular complexes from sequences. It outputs full-atom structures for proteins, ligands, and glycans with confidence scores including per-residue pLDDT, interface pTM (ipTM), and overall structure confidence.

**Why is this important?**
Understanding how proteins interact with small molecules is critical for:
- Drug design and virtual screening (predicting binding poses)
- Validating designed protein-ligand interactions
- Modeling glycoprotein structures with attached carbohydrates
- Predicting binding site conformations
- Screening compound libraries against protein targets

**Scientific foundation:**
Chai1 uses a modern deep learning architecture combining:
1. **ESM-2 embeddings** (optional): Pre-trained protein language model representations that capture evolutionary and structural patterns from millions of protein sequences.
2. **Diffusion-based structure generation**: A generative modeling approach that iteratively denoises random coordinates into realistic 3D structures, naturally handling the multi-modal nature of biomolecular complexes.
3. **Trunk network**: An attention-based architecture that processes sequence features through multiple recycling passes for iterative refinement.

Confidence metrics include:
- **pLDDT** (predicted Local Distance Difference Test): Per-residue confidence score (0-100), where >90 indicates high confidence, 70-90 is moderate, and <70 suggests low confidence.
- **pTM** (predicted Template Modeling score): Overall structure accuracy (0-1), where >0.8 indicates high confidence in the global fold.
- **ipTM** (interface pTM): Confidence in inter-chain interfaces (0-1), critical for multi-chain complexes.

## Available tools

In [2]:
display_available_tools("chai1")

- **`run_chai1()`** — Multi-modal structure prediction using Chai1

### `run_chai1`

Chai-1 predicts 3D structures of proteins, ligands, and glycans using a diffusion-based model that optionally integrates ESM protein language model embeddings for improved predictions without requiring an explicit MSA search. The `num_trunk_recycles` parameter controls iterative refinement of the structure representation, while `num_diffn_timesteps` governs the resolution of the denoising process. Multiple independent diffusion samples can be generated via `num_diffn_samples`, with the best sample returned by confidence score. Chai-1 caps each complex at 2,048 tokens (1 per amino acid, 1 per heavy atom for ligands and glycans) and does not support DNA or RNA inputs.

In [3]:
from pathlib import Path

from proto_tools import (
    Chai1Config,
    Chai1Input,
    Chain,
    StructurePredictionComplex,
    run_chai1,
)

In [4]:
# Display input docs
display_api_reference("chai1", "input", "run_chai1")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Chai1Input(complexes=[complex])

**Input** — `Chai1Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `list[proto_tools.tools.structure_prediction.shared_data_models.StructurePredictionComplex]` | required | List of complexes to predict structure for containing chains and entity types. |
| `msas` | `dict[str, proto_tools.tools.sequence_alignment.msas.MSA] | None` | `None` | Pre-computed MSAs keyed by sequence. Populated by preprocess() or supplied directly. |

In [5]:
# Display config docs
display_api_reference("chai1", "config", "run_chai1")

# Configure Chai-1
config = Chai1Config(
    verbose=True,
    device="cuda",  # Change to "cpu" if no GPU available
    use_esm_embeddings=True,
    use_msa=False,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    num_diffn_samples=1,
)

**Config** — `Chai1Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on (e.g., 'cuda', 'cpu') |
| `timeout` | `int` | `1200` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `include_pae_matrix` | `bool` | `False` | Attach the full per-residue PAE matrix. |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using ColabFold search |
| `colabfold_search_config` | `proto_tools.tools.sequence_alignment.colabfold_search.colabfold_search.ColabfoldSearchConfig | None` | `None` | Nested configuration for ColabFold MSA search. If None, uses default settings. |
| `use_esm_embeddings` | `bool` | `True` | Whether to use ESM embeddings for improved predictions |
| `num_trunk_recycles` | `int` | `3` | Iterative refinement passes through the trunk network. Higher = more accurate but slower. |
| `num_diffn_timesteps` | `int` | `200` | Denoising steps in the diffusion process. Higher = more refined but slower. |
| `num_diffn_samples` | `int` | `5` | Structure samples per complex; best by confidence is kept. Higher = more thorough but slower. |
| `num_trunk_samples` | `int` | `1` | Independent trunk forward passes per diffusion sample (adds sample diversity). |
| `low_memory` | `bool` | `True` | Stream features per sample to reduce peak GPU memory at the cost of speed. |
| `recycle_msa_subsample` | `int` | `0` | Randomly subsample the MSA across recycles for diversity. 0 disables. |

In [6]:
# Run structure prediction
result = run_chai1(inputs, config)

Folding structures (Chai-1):   0%|          | 0/1 [00:00<?, ?complex/s]

In [7]:
# Display output docs
display_api_reference("chai1", "output", "run_chai1")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Average pLDDT: {mfng_structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.metrics.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.metrics.iptm:.3f}")

**Output** — `Chai1Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `list[proto_tools.entities.structures.structure.Structure]` | required | List of predicted structures |

  Number of chains: 2
  Protein length: 384 residues
  Average pLDDT: 0.838
  pTM score: 0.844
  ipTM score: 0.612


#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain. You can rotate, zoom, and inspect the binding interface between MfnG and L-tyrosine.

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [8]:
mfng_structure.visualize(color_by='chain')

## Export Results

Predicted structures can be exported to PDB or mmCIF format for further analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT confidence scores for per-residue visualization.

In [9]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'mfng_ligand_complex.pdb'}")

Structure exported to example_output/mfng_ligand_complex.pdb


/large_storage/hielab/bviggiano/codebases/evo-design/proto-tools/proto_tools/entities/structures/structure.py:569: UserWarning: CIF→PDB conversion: 1 residue name(s) exceed PDB's 3-character limit and will be truncated (e.g., ['LIG2']).
  return convert_cif_str_to_pdb_str(self.structure)
